<a href="https://colab.research.google.com/github/vencov/FAV_upsidedown/blob/main/TutOEME/EX_chirp_time_averaging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time-domain averaging of chirp-evoked ear-canal responses

When a chirp stimulus is repeated many times and the ear-canal recording
is cut into one epoch per chirp, the **stimulus-evoked response** is the
same (phase-locked) in every epoch, while the **noise** (physiological,
electronic) is essentially independent from repeat to repeat. Averaging
the epochs together cancels the noise (its amplitude shrinks roughly as
1/√N) while leaving the evoked response untouched — this is the standard
trick used to pull small otoacoustic/ear-canal responses out of a noisy
recording.

This notebook:
1. loads a recording consisting of `Nchirps` repeated chirps back to back,
2. cuts it into individual chirp-length epochs,
3. plots several raw (single-trial) epochs so you can see how noisy any
   *one* chirp response looks,
4. averages across epochs and overlays the averaged response on a single
   raw epoch, so you can see directly how much the noise is reduced.


## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

plt.rcParams['figure.dpi'] = 100


## 2. Load the data

Set `USE_DEMO_DATA = True` to try the notebook with a synthetic
(simulated) recording — useful for testing the notebook itself before
you have real data, or for a first pass through the exercise.

Set `USE_DEMO_DATA = False` to use your own recording. Your `.mat` file
should contain the same fields as in the lab recordings:
- `y1all`, `y2all` — the two recorded ear-canal channels (one long
  vector each, containing all `Nchirps` repeats back to back)
- `Nsamp` — number of samples per chirp
- `fsamp` — sampling rate (Hz)
- `latSC` — sound-card / system latency to strip from the start (samples)

`Nchirps` (how many chirps were recorded) is **not** stored in the file —
set it below to match your recording protocol.


In [ ]:
USE_DEMO_DATA = False  # <-- set to False to load your own .mat file
from google.colab import drive
drive.mount('/content/drive/MyDrive/Colab\'Notebooks/')

if USE_DEMO_DATA:
    # ---- synthetic demo recording: a short chirp buried in noise, repeated Nchirps times ----
    from scipy.signal import chirp as make_chirp

    fsamp = 48000.0
    Nsamp = 2000
    latSC = 50
    Nchirps = 300

    rng = np.random.default_rng(0)
    t_chirp = np.arange(Nsamp) / fsamp
    true_response = 0.05 * make_chirp(t_chirp, f0=500, f1=8000, t1=t_chirp[-1], method='linear')

    noise_amp = 0.5
    y1all = np.concatenate([
        np.zeros(latSC),
        np.tile(true_response, Nchirps) + noise_amp * rng.standard_normal(Nchirps * Nsamp)
    ])
    y2all = np.concatenate([
        np.zeros(latSC),
        np.tile(true_response, Nchirps) + noise_amp * rng.standard_normal(Nchirps * Nsamp)
    ])
    print(f"Using synthetic demo data: Nchirps={Nchirps}, Nsamp={Nsamp}, fsamp={fsamp} Hz")

else:
    fname = 'InDeadEarData.mat'  # <-- change to your file name

    if not os.path.exists(fname):
        try:
            from google.colab import files
            print(f"'{fname}' not found in the working directory — please upload it:")
            uploaded = files.upload()
            fname = list(uploaded.keys())[0]
        except ImportError:
            raise FileNotFoundError(
                f"'{fname}' not found, and google.colab is not available to upload it "
                "interactively. Place the file in the working directory, or run this in Colab."
            )

    data = loadmat(fname)
    y1all = data['y1all'][0]
    y2all = data['y2all'][0]
    Nsamp = int(data['Nsamp'][0][0])
    fsamp = float(data['fsamp'][0][0])
    latSC = int(data['latSC'][0][0])
    Nchirps = 300  # <-- set this to match your recording protocol

    print(f"Loaded '{fname}': Nchirps={Nchirps}, Nsamp={Nsamp}, fsamp={fsamp} Hz")


MessageError: Error: credential propagation was unsuccessful

## 3. Cut the recording into individual chirp epochs

Strip the sound-card latency from the start, then reshape the long
recording into an array of shape `(Nchirps, Nsamp)` — one row per chirp
repeat.


In [ ]:
def epoch_chirps(y, latSC, Nsamp, Nchirps):
    '''Strip the initial latency and cut y into Nchirps epochs of Nsamp samples each.'''
    y_stripped = y[latSC:]
    return np.reshape(y_stripped[:Nchirps * Nsamp], (Nchirps, Nsamp))


y1_epochs = epoch_chirps(y1all, latSC, Nsamp, Nchirps)  # shape (Nchirps, Nsamp)
y2_epochs = epoch_chirps(y2all, latSC, Nsamp, Nchirps)

t_ms = np.arange(Nsamp) / fsamp * 1000  # time axis for one epoch, in ms

print(f"y1_epochs shape: {y1_epochs.shape}   (Nchirps x Nsamp)")


## 4. A few individual (single-trial) chirp responses

Before averaging, look at several individual chirps on their own — this
is what the raw, unaveraged response actually looks like.


In [ ]:
chirps_to_show = np.arange(10, 18)  # 8 example chirps

fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for idx in chirps_to_show:
    axs[0].plot(t_ms, y1_epochs[idx], alpha=0.7, linewidth=0.8)
    axs[1].plot(t_ms, y2_epochs[idx], alpha=0.7, linewidth=0.8)

axs[0].set_title('Channel 1 — individual chirp responses')
axs[1].set_title('Channel 2 — individual chirp responses')
for ax in axs:
    ax.set_xlabel('Time (ms)')
axs[0].set_ylabel('Amplitude (a.u.)')
plt.tight_layout()
plt.show()


## 5. Average across chirps, and compare to a single raw chirp

The first few chirps are skipped (`Nchskip`) to avoid any onset
transient/adaptation at the very start of the stimulus train. The rest
are averaged together and overlaid on one single (unaveraged) chirp
response, so the noise reduction from averaging is directly visible.


In [ ]:
Nchskip = 10  # skip the first few chirps (onset transient), same as in the lab script

y1_avg = np.mean(y1_epochs[Nchskip:], axis=0)
y2_avg = np.mean(y2_epochs[Nchskip:], axis=0)

single_idx = Nchskip  # the first chirp actually used in the average, for a fair comparison

fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

axs[0].plot(t_ms, y1_epochs[single_idx], color='gray', alpha=0.7,
            label='Single chirp (no averaging)')
axs[0].plot(t_ms, y1_avg, color='C0', linewidth=1.8,
            label=f'Averaged over {Nchirps - Nchskip} chirps')
axs[0].set_title('Channel 1')

axs[1].plot(t_ms, y2_epochs[single_idx], color='gray', alpha=0.7,
            label='Single chirp (no averaging)')
axs[1].plot(t_ms, y2_avg, color='C1', linewidth=1.8,
            label=f'Averaged over {Nchirps - Nchskip} chirps')
axs[1].set_title('Channel 2')

for ax in axs:
    ax.set_xlabel('Time (ms)')
    ax.legend(loc='upper right', fontsize=8)
axs[0].set_ylabel('Amplitude (a.u.)')
plt.tight_layout()
plt.show()
